# 04 - Tier 2: the agent that uses tools

Tier 1 answered from memory and made numbers up. Tier 2 runs real queries.

**This is Abdelateef's Tier 2 design**, written in the same plain style as
the other notebooks so it plugs straight into the Tier 3 checker.

How it works:

1. **Plan** - the model lists the steps, each using one tool
2. **Run** - for every step we ask for just that one thing and run it
3. **Answer** - the model writes the answer from the rows that came back

Three ideas make it much harder to go wrong than a single prompt would:

| Idea | Why |
|---|---|
| The model never writes Python | it picks an operation, we do the sum |
| SQL is parsed, not word-matched | a column called `updated_at` contains "update" |
| Fields and filters read off the SQL | the model makes them up otherwise |

Everything it does is recorded in a log. Notebook 05 checks the answer
against that log.


## Setup

`sqlglot` is new - it parses SQL. `pip install sqlglot`


In [1]:
import json
import re
import time

import duckdb
import sqlglot
from sqlglot import expressions as exp

DB = "../data/processed/evidenceiq.duckdb"
MODEL = "gemma3:4b"          # Abdelateef used gemma3:12b - use what you have
TOLERANCE = 0.01


def execution_feedback(answer):
    """Repair an unsuccessful analysis even when it contains no numeric claims."""
    if answer.get('route') in ('query_pattern', 'rules_refusal'):
        return ''
    calls = answer.get('log') or []
    successful = [c for c in calls if c.get('ok') and c.get('tool') in ('run_sql', 'run_python')]
    failure = answer.get('failure')
    if successful and not failure:
        return ''
    failed = [c for c in calls if not c.get('ok') and c.get('tool') in ('run_sql', 'run_python')]
    messages = []
    if failed:
        # Keep the most recent error and the actual attempted query together.
        last = failed[-1]
        messages.append('The previous analysis failed. Do not repeat this query unchanged:\n'
                        + str(last.get('code', '')) + '\nReason: ' + str(last.get('error', 'Unknown tool error')))
    elif failure:
        messages.append('The previous analysis failed: ' + str(failure))
    elif not successful and not answer.get('claims'):
        messages.append('No usable evidence was produced. Plan a read-only query that returns the requested measurements and labels.')
    return '\n'.join(messages)


def tier2_with_retries(question, max_retries=2):
    """Optional equal-budget control: execution feedback only, no verification."""
    from copy import deepcopy
    start = time.time()
    history, usage = [], []
    feedback = ''
    for attempt in range(max_retries + 1):
        answer = tier2(question, [], feedback)
        usage.append(answer.get('usage', {}))
        feedback = execution_feedback(answer)
        history.append({'attempt': attempt, 'answer': deepcopy(answer), 'feedback': feedback})
        if not feedback:
            break
    answer['attempts'] = history
    answer['selected_attempt'] = attempt
    answer['retries'] = attempt
    answer['usage'] = model_usage(usage)
    answer['seconds'] = round(time.time() - start, 2)
    answer['control'] = 'execution_retries_without_verification'
    return answer



def query_guidance(question):
    """Highlight the relevant business rules without selecting SQL or answers."""
    text = (question or '').lower()
    tips = []
    if 'revenue' in text and not re.search(r'product|stock code|item', text):
        tips.append('Use the overall revenue population: exclude cancellations only; no product, outlier or quantity filters.')
    if re.search(r'\bcountr|\buk\b|united kingdom', text):
        tips.append("Country is already on sales. Use 'United Kingdom' for UK; keep guest purchases. Avoid customer joins.")
    if re.search(r'\bcustomers?\b', text):
        tips.append('Use sales, not all-date dim_customer totals. Group rankings by customer_id and exclude NULL customer_id.')
    if re.search(r'\borders?\b', text):
        tips.append('Count DISTINCT invoice_no; exclude cancellations and keep non-product lines.')
    if re.search(r'\bshare\b|percentage.*(?:uk|countr)|(?:uk|countr).*percentage', text):
        tips.append('Return both numerator and denominator; the denominator must include all requested countries, not just the numerator country.')
    if re.search(r'month|trading day|january|february|march|april|may|june|july|august|september|october|november|december', text):
        tips.append('Use dim_month. Include each requested period, revenue, trading_days and is_complete_month; use no sales-only filters.')
        if 'per trading day' in text or 'per day' in text:
            tips.append('Compute revenue / trading_days for EACH period, then compare these daily values as well as total revenues.')
        if re.search(r'highest|lowest|most|fewest|maximum|minimum', text):
            tips.append('Return every tied extreme. For revenue rankings use only complete months; for trading-day rankings include all months unless restricted.')
    return '\n'.join('- ' + tip for tip in tips)


def country_scope_problem(sql, question, countries=None):
    """Check explicitly named countries against SQL populations, never answers.

    The vocabulary comes from this database, so the rule applies to any market.
    Country shares may have an unrestricted denominator; their numerator must
    still select the requested country. This is a bounded intent guard.
    """
    if countries is None:
        try:
            with duckdb.connect(DB, read_only=True, config={'enable_external_access': 'false'}) as con:
                countries = [row[0] for row in con.execute('SELECT DISTINCT country FROM sales').fetchall()]
        except Exception:
            return None  # Database availability is reported by run_sql itself.
    vocabulary = {str(country).casefold(): str(country) for country in countries if country is not None}
    aliases = {'uk': 'United Kingdom', 'u.k.': 'United Kingdom'}
    names = dict(vocabulary)
    names.update({alias: country for alias, country in aliases.items() if country.casefold() in vocabulary})
    text = str(question or '').casefold()
    included, excluded = set(), set()
    country_names = '|'.join(re.escape(name) for name in sorted(names, key=len, reverse=True))
    for name, country in names.items():
        for mention in re.finditer(r'(?<!\w)' + re.escape(name) + r'(?!\w)', text):
            prefix = text[max(0, mention.start()-180):mention.start()]
            negative = bool(re.search(r'(?:outside(?:\s+of)?|excluding|except(?:\s+for)?|other\s+than|not\s+(?:from|in))\s+(?:the\s+)?(?:(?:' + country_names + r')\s*(?:,|and|or)\s*)*$', prefix))
            (excluded if negative else included).add(country)
    if not included and not excluded:
        return None
    universe = set(vocabulary.values())
    tree = sqlglot.parse_one(sql, read='duckdb')

    def domain(predicate):
        if isinstance(predicate, (exp.Where, exp.Having, exp.Paren)):
            return domain(predicate.this)
        if isinstance(predicate, exp.And):
            return domain(predicate.this) & domain(predicate.expression)
        if isinstance(predicate, exp.Or):
            return domain(predicate.this) | domain(predicate.expression)
        if isinstance(predicate, exp.Not):
            columns = list(predicate.this.find_all(exp.Column))
            if columns and all(column.name.lower() == 'country' for column in columns):
                return universe - domain(predicate.this)
            return universe
        if isinstance(predicate, (exp.EQ, exp.NEQ)):
            left, right = predicate.this, predicate.expression
            if isinstance(right, exp.Column):
                left, right = right, left
            if isinstance(left, exp.Column) and left.name.lower() == 'country' and isinstance(right, exp.Literal) and right.is_string:
                chosen = {vocabulary.get(str(right.this).casefold(), str(right.this))}
                return universe - chosen if isinstance(predicate, exp.NEQ) else chosen
        if isinstance(predicate, exp.In) and isinstance(predicate.this, exp.Column) and predicate.this.name.lower() == 'country' and predicate.expressions:
            if all(isinstance(value, exp.Literal) and value.is_string for value in predicate.expressions):
                return {vocabulary.get(str(value.this).casefold(), str(value.this)) for value in predicate.expressions}
        return universe

    restrictions = [domain(predicate) for predicate in tree.find_all(exp.Where, exp.Having)]
    restrictions = [values for values in restrictions if values != universe]
    country_cases = [case for case in tree.find_all(exp.If)
                     if any(column.name.lower() == 'country' for column in case.this.find_all(exp.Column))]
    case_restrictions = [domain(case.this) for case in country_cases]
    candidates = restrictions or case_restrictions
    share_question = bool(re.search(r'\bshare\b|\bproportion\b|\b(?:percentage|percent)\b.*\b(?:from|of)\b', text))
    country_filtered = any(column.name.lower() == 'country' for predicate in tree.find_all(exp.Where, exp.Having) for column in predicate.find_all(exp.Column))
    if share_question and not candidates and not country_filtered:
        return None  # A separate query may fetch the unrestricted denominator.
    allowed = (included or universe) - excluded
    expected = ', '.join(sorted(included)) if included else 'all countries except ' + ', '.join(sorted(excluded))
    if not candidates or any(not values or not values <= allowed for values in candidates):
        return ('The question requests ' + expected + ', but the SQL country population differs or is missing. '
                'Use the requested full country names in WHERE/IN; comparisons may query each requested country separately. '
                'For a share, select the requested country in CASE WHEN for the numerator and keep the all-country denominator.')
    # A WHERE country restriction scopes every measurement. Without one, each
    # copied measure needs a country-specific CASE; only shares need a whole.
    if not restrictions and not share_question:
        for aggregate in tree.find_all(exp.AggFunc):
            for column in aggregate.find_all(exp.Column):
                if column.name.lower() not in {'revenue', 'quantity', 'invoice_no', 'customer_id'}:
                    continue
                current, scoped = column, False
                while current is not aggregate and current.parent is not None:
                    parent = current.parent
                    if isinstance(parent, exp.If) and parent.args.get('true') is current and domain(parent.this) <= allowed:
                        scoped = True
                        break
                    current = parent
                if not scoped:
                    return ('A requested-country measure still includes other countries. Restrict it with WHERE country/IN, '
                            'or CASE WHEN country matches THEN measure ELSE 0/NULL END. Keep all-country totals only for a share denominator.')
    return None


## 1. Making the database safe

The agent writes its own SQL, so we have to stop it changing anything.
We parse the query into a tree and look at what it actually does, instead
of searching the text for banned words.


In [5]:
# We never let the agent write to the database. Two safety nets: this check,
# and opening DuckDB with read_only=True.
#
# We parse the SQL into a syntax tree instead of searching for banned words.
# Word matching is easy to fool - a column called "updated_at" contains
# "update" - and it misses things a parser catches for free.

FORBIDDEN = {"insert", "update", "delete", "create", "drop", "alter", "copy",
             "command", "merge", "truncate", "attach", "detach", "install",
             "load", "pragma"}


def check_sql(sql):
    """Parse one read-only query; DuckDB also blocks writes and external files."""
    if not isinstance(sql, str) or not sql.strip():
        return "the query is empty"
    try:
        statements = sqlglot.parse(sql, read="duckdb")
        if len(statements) != 1 or statements[0] is None:
            return "only one SELECT query is allowed"
        tree = statements[0]
        if not isinstance(tree, exp.Query):
            return "only SELECT queries are allowed"
        for node in tree.walk():
            if getattr(node, "key", "").lower() in FORBIDDEN:
                return "forbidden operation: %s" % node.key
    except Exception as error:
        return "could not parse the SQL: %s" % error
    return None


In [3]:
for sql in ["SELECT SUM(revenue) FROM sales",
            "DROP TABLE sales",
            "SELECT 1; DELETE FROM sales",
            "SELECT updated_at FROM sales",       # contains the word "update"
            "UPDATE sales SET revenue = 0"]:
    print("%-45s -> %s" % (sql[:45], check_sql(sql) or "safe"))


SELECT SUM(revenue) FROM sales                -> safe
DROP TABLE sales                              -> only SELECT queries are allowed
SELECT 1; DELETE FROM sales                   -> only one SELECT query is allowed
SELECT updated_at FROM sales                  -> safe
UPDATE sales SET revenue = 0                  -> only SELECT queries are allowed


Note the fourth one. A word-matching check would have blocked it.


## 2. The log

`add_to_log` records a call, `numbers_in_log` pulls out every number the
tools returned, and `show_results` turns the log back into text for the
next prompt.


In [9]:
def add_to_log(log, tool, code, ok, rows=None, error=None):
    """Every tool call goes in the log. The verifier only trusts what's here."""
    call = {"n": len(log), "tool": tool, "code": code,
            "ok": ok, "rows": rows or [], "error": error}
    log.append(call)
    return call


def numbers_in_log(log):
    """SQL is the root evidence. Calculations must trace back to earlier evidence."""
    import math
    from decimal import Decimal
    found = []
    for call in log or []:
        if not isinstance(call, dict) or not call.get("ok"):
            continue
        for row in call.get("rows") or []:
            if not isinstance(row, dict):
                continue
            if call.get("tool") == "run_python":
                try:
                    inputs = row.get("inputs")
                    result, _ = calculate(row.get("operation"), inputs)
                    # Matching arbitrary model inputs would turn invented values into evidence.
                    if not inputs or not all(any(math.isclose(float(x), v, rel_tol=1e-6, abs_tol=.01)
                                                 for _, v in found) for x in inputs):
                        continue
                    if not math.isclose(float(row["value"]), result, rel_tol=1e-6, abs_tol=.0001):
                        continue
                    values = [row["value"]]
                except (TypeError, ValueError, KeyError, OverflowError):
                    continue
            elif call.get("tool") == "run_sql":
                values = [value for key, value in row.items() if key not in ('customer_id', 'stock_code', 'invoice_no')]
            else:
                continue  # chart metadata is not numerical evidence
            for value in values:
                if isinstance(value, (int, float, Decimal)) and not isinstance(value, bool):
                    if math.isfinite(float(value)):
                        found.append((call.get("n"), float(value)))
    return found


def show_results(log, start=0):
    """Turn the log into text we can paste into the next prompt."""
    if len(log) <= start:
        return "(no queries were run)"
    text = ""
    for call in log[start:]:
        text += "[call %d] %s\n%s\n" % (call["n"], call["tool"], str(call["code"]).strip())
        text += "-> %s\n\n" % (json.dumps(call["rows"][:20]) if call["ok"]
                                else "FAILED: " + str(call["error"]))
    return text


## 3. Reading the query back

Two of the seven things we must show the user are the fields and filters
used. We read them off the SQL rather than asking the model.


In [11]:
def sql_metadata(sql):
    """Which columns and filters did this query actually use?

    Two of the seven things we have to show the user are the fields and the
    filters. We read them off the query itself rather than asking the model,
    because the model will happily make them up.

    Aliases are skipped: in "SUM(revenue) AS total_revenue" the real field is
    revenue, not total_revenue.
    """
    try:
        tree = sqlglot.parse_one(sql, read="duckdb")
    except Exception:
        return [], []

    aliases = {a.alias for a in tree.find_all(exp.Alias) if a.alias}

    fields = []
    for column in tree.find_all(exp.Column):
        name = column.name
        if name and name not in aliases and name not in fields:
            fields.append(name)

    filters = []
    for where in tree.find_all(exp.Where):
        text = where.this.sql(dialect="duckdb")
        if text not in filters:
            filters.append(text)

    return fields, filters


In [6]:
fields, filters = sql_metadata(
    "SELECT stock_code, ROUND(SUM(revenue),2) AS total FROM sales "
    "WHERE NOT is_cancellation AND year(invoice_date)=2011 GROUP BY stock_code")

print("fields :", fields)
print("filters:", filters)


fields : ['stock_code', 'revenue', 'is_cancellation', 'invoice_date']
filters: ['NOT is_cancellation AND YEAR(invoice_date) = 2011']


`total` is missing from the fields on purpose - it is an alias we invented,
not a column in the data.


## 4. The tools


### run_sql


In [16]:
def json_safe(value):
    """Keep decimal aggregates numeric and dates readable in the evidence log."""
    from decimal import Decimal
    if isinstance(value, Decimal):
        return float(value)
    if hasattr(value, "isoformat"):
        return value.isoformat()
    if hasattr(value, "item"):
        return value.item()
    return value


def query_scope_problem(sql, question):
    """Check known business populations and date scopes, not arbitrary SQL meaning.

    Case/filtered aggregates may restrict dates without an outer WHERE. Their
    conditions count as date restrictions, but a date in an alias or SELECT
    literal does not. These guards supplement execution and evidence checks.
    """
    tree = sqlglot.parse_one(sql, read='duckdb')
    contract = question_contract(question)
    if contract:
        expected = sqlglot.parse_one(contract['sql'], read='duckdb')
        if tree != expected:
            return 'This question has a checked KPI query pattern. Use: ' + contract['sql']
        return None

    country_problem = country_scope_problem(sql, question)
    if country_problem:
        return country_problem

    text = (question or '').lower()
    years = set(re.findall(r'\b(?:19|20)\d{2}\b', text))
    tables = {table.name.lower() for table in tree.find_all(exp.Table)}
    # Month summaries have one row per month; sales has many invoice lines.
    # Joining without the month key repeats the entire fact table, and summing
    # month-level measures after a line-level join multiplies those measures.
    fact_relations, raw_fact_relations = {'sales'}, {'sales'}
    month_relations = {'dim_month'}

    def source_kind(source):
        if isinstance(source, exp.Table):
            name = source.name.lower()
            return name in fact_relations, name in raw_fact_relations, name in month_relations
        if isinstance(source, exp.Subquery):
            return select_kind(source.this)
        return False, False, False

    def select_kind(select):
        if not isinstance(select, exp.Select):
            return False, False, False
        source = select.args.get('from_')
        relations = ([source.this] if source is not None else []) + [join.this for join in select.args.get('joins', [])]
        kinds = [source_kind(relation) for relation in relations]
        fact = any(kind[0] for kind in kinds)
        group = select.args.get('group')
        # Only one row per invoice_month removes the line-level duplication.
        one_row_per_month = bool(group and len(group.expressions) == 1
                                 and isinstance(group.expressions[0], exp.Column)
                                 and group.expressions[0].name.lower() == 'invoice_month')
        return fact, fact and not one_row_per_month, any(kind[2] for kind in kinds)

    for cte in tree.find_all(exp.CTE):
        fact, raw_fact, month = select_kind(cte.this)
        name = cte.alias_or_name.lower()
        if fact:
            fact_relations.add(name)
        if raw_fact:
            raw_fact_relations.add(name)
        if month:
            month_relations.add(name)

    def links_month(predicate, fact_aliases, month_alias):
        if isinstance(predicate, (exp.Where, exp.Paren)):
            return links_month(predicate.this, fact_aliases, month_alias)
        if isinstance(predicate, exp.And):
            return any(links_month(part, fact_aliases, month_alias) for part in (predicate.this, predicate.expression))
        if isinstance(predicate, exp.Or):
            return all(links_month(part, fact_aliases, month_alias) for part in (predicate.this, predicate.expression))
        if isinstance(predicate, exp.EQ):
            left, right = predicate.this, predicate.expression
            if isinstance(left, exp.Column) and isinstance(right, exp.Column) and left.name.lower() == right.name.lower() == 'invoice_month':
                return ((left.table.lower() in fact_aliases and right.table.lower() == month_alias)
                        or (right.table.lower() in fact_aliases and left.table.lower() == month_alias))
        return False

    for select in tree.find_all(exp.Select):
        source = select.args.get('from_')
        relations = ([source.this] if source is not None else []) + [join.this for join in select.args.get('joins', [])]
        classified = [(relation.alias_or_name.lower(), source_kind(relation)) for relation in relations]
        fact_aliases = {alias for alias, kind in classified if kind[0]}
        month_aliases = {alias for alias, kind in classified if kind[2]}
        if not fact_aliases or not month_aliases:
            continue
        clauses = ([select.args['where']] if select.args.get('where') is not None else [])
        clauses += [join.args['on'] for join in select.args.get('joins', []) if join.args.get('on') is not None]
        for month_alias in month_aliases:
            using_month = any((join.this.alias_or_name.lower() == month_alias
                               or (join.this.alias_or_name.lower() in fact_aliases and source is not None and source.this.alias_or_name.lower() == month_alias))
                              and any(identifier.name.lower() == 'invoice_month' for identifier in join.args.get('using') or [])
                              for join in select.args.get('joins', []))
            if not using_month and not any(links_month(clause, fact_aliases, month_alias) for clause in clauses):
                return ('A sales/month join must link the matching month: sales_alias.invoice_month = month_alias.invoice_month. '
                        'ON 1=1 or a date filter only on dim_month repeats sales from unrelated months. '
                        'For overall monthly revenue/per-day comparisons, remove sales and query dim_month directly.')
        if any(kind[1] for _, kind in classified):
            for aggregate in select.find_all(exp.Sum, exp.Avg):
                if aggregate.find_ancestor(exp.Select) is not select:
                    continue
                if any(column.name.lower() in {'trading_days', 'net_revenue', 'gross_revenue'}
                       and (not column.table or column.table.lower() in month_aliases)
                       for column in aggregate.find_all(exp.Column)):
                    return ('Do not SUM or AVG dim_month measures after joining invoice lines: this repeats trading_days/monthly revenue once per line. '
                            'For overall monthly analysis use dim_month alone. For filtered sales, group sales by invoice_month first, '
                            'then join that one-row-per-month result to dim_month on invoice_month.')

    wheres = list(tree.find_all(exp.Where))
    filters = ' '.join(where.sql(dialect='duckdb') for where in wheres)
    normalized = re.sub(r'\b\w+\.', '', filters.lower())
    normalized = re.sub(r'\s+', ' ', normalized)
    temporal_columns = {'invoice_date', 'invoice_month'}

    def predicate_years(predicate):
        # A year in one OR branch does not restrict the other branch. Similarly,
        # NOT(year = ...) selects other dates, not the named reporting year.
        if isinstance(predicate, (exp.Where, exp.Paren)):
            return predicate_years(predicate.this)
        if isinstance(predicate, exp.And):
            return predicate_years(predicate.this) | predicate_years(predicate.expression)
        if isinstance(predicate, exp.Or):
            left, right = predicate_years(predicate.this), predicate_years(predicate.expression)
            return left | right if left and right else set()
        if isinstance(predicate, exp.Not):
            return set()
        if not any(col.name.lower() in temporal_columns for col in predicate.find_all(exp.Column)):
            return set()
        return {year for literal in predicate.find_all(exp.Literal)
                for year in re.findall(r'\b(?:19|20)\d{2}\b', str(literal.this))}

    # An aggregate FILTER affects only that aggregate, never its neighbours.
    where_years = set().union(*(predicate_years(where) for where in wheres
                                if not isinstance(where.parent, exp.Filter)))
    conditional_years = set().union(*(predicate_years(case.args.get('this'))
                                      for case in tree.find_all(exp.If)
                                      if case.args.get('this') is not None))
    filter_years = set().union(*(predicate_years(where) for where in wheres
                                 if isinstance(where.parent, exp.Filter)))
    if tables.intersection({'sales', 'dim_month'}) and years:
        if not years <= where_years | conditional_years | filter_years:
            return ('Restrict every requested reporting year using invoice_date on sales or invoice_month on dim_month. '
                    'A selected year label is not a date filter; keep all requested periods in WHERE/IN or CASE conditions.')
        if not years <= where_years:
            # Without an outer date scope, every aggregate of a measure needs a
            # conditional date scope. Do not let one CASE launder an all-time SUM.
            for aggregate in tree.find_all(exp.AggFunc):
                measures = [col for col in aggregate.find_all(exp.Column)
                            if col.name.lower() in {'revenue', 'net_revenue', 'gross_revenue', 'quantity', 'invoice_no'}]
                if not measures:
                    continue
                local_filter = (predicate_years(aggregate.parent.args['expression'])
                                if isinstance(aggregate.parent, exp.Filter) else set())
                def conditional_measure_scoped(column):
                    # Each measure must sit in a date-restricted THEN branch.
                    # ELSE revenue would reintroduce rows outside the period.
                    current = column
                    while current is not aggregate and current.parent is not None:
                        parent = current.parent
                        if (isinstance(parent, exp.If) and parent.args.get('true') is current
                                and predicate_years(parent.this) & years):
                            return True
                        current = parent
                    return False
                if not (local_filter & years or all(conditional_measure_scoped(col) for col in measures)):
                    return ('An aggregate still covers all dates. Put the requested dates in WHERE, '
                            'use FILTER for that aggregate, or restrict every measured value with '
                            'CASE WHEN date condition THEN measure ELSE 0/NULL END.')

    for predicate in tree.find_all(exp.Predicate):
        if any(col.name.lower() == 'country' for col in predicate.find_all(exp.Column)):
            if any(lit.is_string and str(lit.this).strip().lower() in {'uk', 'u.k.', 'great britain'}
                   for lit in predicate.find_all(exp.Literal)):
                return "Use country = 'United Kingdom' (or IN/<> with that full name); the database has no country named 'UK'."

    explicit_exclusion = bool(re.search(r'(?:exclud\w*|without)\s+(?:the\s+)?cancellations?|cancellations?\s+(?:are\s+)?excluded', text))
    revenue_question = 'revenue' in text and (explicit_exclusion or not re.search(r'cancel|return', text))
    product_scope = bool(re.search(r'product|stock code|item', text))
    customer_ranking = bool(re.search(r'(?:top|which).*customers', text))
    identified = bool(re.search(r'customer_id IS NOT NULL|NOT customer_id IS NULL', normalized, re.I))
    grouped_customers = any(col.name.lower() == 'customer_id' for group in tree.find_all(exp.Group)
                            for col in group.find_all(exp.Column))
    def requires_boolean(predicate, column, value, negated=False):
        # This tests implication: can a matching row have the opposite flag?
        # Under NOT, De Morgan swaps AND/OR. Inverting only the desired value
        # would wrongly accept NOT(is_cancellation AND country = ...).
        if isinstance(predicate, exp.Paren):
            return requires_boolean(predicate.this, column, value, negated)
        if isinstance(predicate, exp.Not):
            return requires_boolean(predicate.this, column, value, not negated)
        if isinstance(predicate, (exp.And, exp.Or)):
            combine = all if (isinstance(predicate, exp.Or) != negated) else any
            return combine(requires_boolean(part, column, value, negated)
                           for part in (predicate.this, predicate.expression))
        if isinstance(predicate, exp.Column):
            return predicate.name.lower() == column and value is not negated
        if isinstance(predicate, (exp.EQ, exp.Is)):
            left, right = predicate.this, predicate.expression
            expected = not value if negated else value
            return ((isinstance(left, exp.Column) and left.name.lower() == column
                     and isinstance(right, exp.Boolean) and right.this is expected)
                    or (isinstance(right, exp.Column) and right.name.lower() == column
                        and isinstance(left, exp.Boolean) and left.this is expected))
        return False

    cancellation_excluded = any(requires_boolean(where.this, 'is_cancellation', False) for where in wheres)
    extra_exclusions = bool(re.search(r'is_product|is_outlier|quantity', normalized)) and not product_scope

    if years and tables.intersection({'dim_customer', 'dim_product'}) and 'sales' not in tables:
        return 'dim_customer and dim_product cover all dates. Query sales with the requested year filter; do not filter first_order or last_order to estimate annual totals.'
    if revenue_question and 'dim_customer' in tables and 'sales' not in tables:
        return 'dim_customer.net_revenue includes cancellations. Query SUM(revenue) from sales WHERE NOT is_cancellation for customer revenue.'
    if 'sales' in tables and customer_ranking and (not identified or not grouped_customers):
        return ('Customer rankings require customer_id IS NOT NULL and GROUP BY customer_id on sales. '
                'Group by customer_id, not country; keep the requested date and revenue/order metric.')
    if 'sales' in tables and revenue_question and 'dim_customer' in tables and 'customer' not in text:
        return ('For overall/country revenue use sales directly: country is already present. '
                'A customer join can drop guest purchases or repeat customer totals; remove that join.')
    if re.search(r'\borders?\b', text) and 'sales' in tables:
        distinct_orders = any(isinstance(count.this, exp.Distinct)
                              and any(col.name.lower() == 'invoice_no' for col in count.find_all(exp.Column))
                              for count in tree.find_all(exp.Count))
        if not distinct_orders or not cancellation_excluded or extra_exclusions:
            return ('Completed orders require COUNT(DISTINCT invoice_no) and WHERE NOT is_cancellation. '
                    'For customer rankings also require customer_id IS NOT NULL. '
                    'Remove product, outlier and quantity exclusions unless the question asks for that population. '
                    'Keep the requested customer/date filters.')
    if revenue_question and 'sales' in tables:
        if not cancellation_excluded:
            return 'Revenue must exclude cancellations: add WHERE NOT is_cancellation (or AND NOT is_cancellation to the existing WHERE).'
        if extra_exclusions:
            return ('Overall/country/customer revenue must not use product-only, outlier or quantity exclusions. '
                    'Remove is_product, is_outlier and quantity filters; keep NOT is_cancellation and the requested country/customer/date filters.')
    if revenue_question and 'dim_month' in tables:
        if any(col.name.lower() == 'gross_revenue' for col in tree.find_all(exp.Column)):
            return 'Use dim_month.net_revenue for revenue excluding cancellations; gross_revenue includes cancellations.'
        monthly_rank = bool(re.search(r'month.*(?:highest|lowest)|(?:highest|lowest).*month', text))
        if monthly_rank and not any(requires_boolean(where.this, 'is_complete_month', True) for where in wheres):
            return 'Monthly revenue rankings require WHERE is_complete_month. Select invoice_month, net_revenue and trading_days; include all tied winners.'
    return None


def run_sql(sql, log, question=None):
    """Run a bounded SELECT, close on errors, and never silently truncate evidence."""
    from threading import Timer
    problem = check_sql(sql)
    if not problem and question:
        problem = query_scope_problem(sql, question)
    if problem:
        return add_to_log(log, "run_sql", sql, False, error=problem)
    con = None
    timer = None
    try:
        con = duckdb.connect(DB, read_only=True, config={"enable_external_access": "false"})
        # A valid SELECT can still be expensive. Interrupt runaway model queries.
        timer = Timer(20, con.interrupt)
        timer.start()
        cursor = con.execute(sql)
        columns = [d[0] for d in cursor.description]
        if len(columns) != len(set(columns)):
            raise ValueError("Give every result column a unique alias.")
        result = cursor.fetchmany(51)
        if len(result) > 50:
            raise ValueError("More than 50 rows. Aggregate the result or add an explicit LIMIT 50.")
        rows = [{c: json_safe(v) for c, v in zip(columns, row)} for row in result]
        return add_to_log(log, "run_sql", sql, True, rows)
    except Exception as error:
        return add_to_log(log, "run_sql", sql, False, error=str(error))
    finally:
        if timer:
            timer.cancel()
            timer.join()
        if con:
            con.close()


In [8]:
log = []

run_sql("SELECT ROUND(SUM(revenue),2) AS revenue FROM sales "
        "WHERE NOT is_cancellation AND year(invoice_date)=2011", log)

log[0]


Out[8]: 
{'n': 0,
 'tool': 'run_sql',
 'code': 'SELECT ROUND(SUM(revenue),2) AS revenue FROM sales WHERE NOT is_cancellation AND year(invoice_date)=2011',
 'ok': True,
 'rows': [{'revenue': 9809614.01}],
 'error': None}


That should be **9,809,614.01**, the same as q01 in our benchmark.


### run_python

The model does not write code. It picks an operation and hands us the
numbers, and we do the arithmetic. Safer, and easier to check later -
the operation and its inputs end up in the log.


In [20]:
# The model does NOT get to write Python. It picks an operation from this
# list and gives us the numbers; we do the arithmetic ourselves.
#
# This is stricter than letting it write code, and it is also easier to check:
# the operation and its inputs are recorded in the log, so Tier 3 can redo the
# sum without trusting anything the model said.

OPERATIONS = ["pct_change", "difference", "ratio", "share", "sum", "mean"]


def calculate(operation, values):
    """Do the arithmetic. Returns (result, how_we_worked_it_out)."""
    if operation == "pct_change":
        if len(values) != 2:
            raise ValueError("pct_change needs [new, old]")
        new, old = values
        if old == 0:
            raise ValueError("cannot work out a percentage change from zero")
        return (new - old) / old * 100, "(%s - %s) / %s * 100" % (new, old, old)

    if operation == "difference":
        if len(values) != 2:
            raise ValueError("difference needs [a, b]")
        return values[0] - values[1], "%s - %s" % (values[0], values[1])

    if operation == "ratio":
        if len(values) != 2 or values[1] == 0:
            raise ValueError("ratio needs [top, bottom] and bottom cannot be zero")
        return values[0] / values[1], "%s / %s" % (values[0], values[1])

    if operation == "share":
        if len(values) != 2 or values[1] == 0:
            raise ValueError("share needs [part, total] and total cannot be zero")
        return values[0] / values[1] * 100, "%s / %s * 100" % (values[0], values[1])

    if operation == "sum":
        if not values:
            raise ValueError("sum needs at least one number")
        return sum(values), " + ".join(str(v) for v in values)

    if operation == "mean":
        if not values:
            raise ValueError("mean needs at least one number")
        return sum(values) / len(values), "mean of %d numbers" % len(values)

    raise ValueError("unknown operation: %s" % operation)


def run_python(request, log):
    """Calculate only with finite numbers already returned by successful tools."""
    import math
    try:
        if not isinstance(request, dict):
            raise ValueError("the calculation request must be an object")
        operation = request.get("operation")
        raw = request.get("values")
        if not isinstance(raw, list) or any(isinstance(v, bool) for v in raw):
            raise ValueError("values must be a list of numbers")
        values = [float(v) for v in raw]
        known = numbers_in_log(log)
        if not all(math.isfinite(v) and any(math.isclose(v, n, rel_tol=1e-6, abs_tol=.01)
                                            for _, n in known) for v in values):
            raise ValueError("every input must come from an earlier successful query or calculation")
        result, code = calculate(operation, values)
        if not math.isfinite(result):
            raise ValueError("the result must be finite")
        unit = request.get("unit") or ("%" if operation in ("pct_change", "share") else None)
        return add_to_log(log, "run_python", code, True,
                          [{"result_name": request.get("result_name") or operation,
                            "value": round(result, 4), "unit": unit,
                            "operation": operation, "inputs": values}])
    except (TypeError, ValueError, OverflowError) as error:
        return add_to_log(log, "run_python", str(request), False, error=str(error))


In [10]:
# Query the inputs first; a calculator cannot establish where numbers came from.
run_sql("SELECT ROUND(net_revenue,2) AS revenue FROM dim_month "
        "WHERE invoice_month IN ('2011-10','2011-11')", log)
run_python({"operation": "pct_change",
            "values": [1503866.78, 1151263.73],
            "result_name": "nov_vs_oct"}, log)

log[-1]["rows"]


Out[10]: 
[{'result_name': 'nov_vs_oct',
  'value': 30.6275,
  'unit': '%',
  'operation': 'pct_change',
  'inputs': [1503866.78, 1151263.73]}]


### make_chart


In [23]:
CHART_TYPES = ["bar", "line", "scatter", "pie", "table", "none"]


def make_chart(spec, log):
    """Validate a real source; infer missing axes only when the choice is clear."""
    problem = None
    spec = dict(spec) if isinstance(spec, dict) else {}
    if spec.get('type') not in CHART_TYPES:
        problem = 'choose a supported chart type'
    elif spec['type'] != 'none':
        source = next((c for c in log if c.get('n') == spec.get('source_tool_call')
                       and c.get('ok') and c.get('tool') == 'run_sql'), None)
        if not source or not source.get('rows'):
            problem = 'chart source must be a successful non-empty SQL result'
        else:
            rows = source['rows']
            numeric = [k for k in rows[0] if all(isinstance(r.get(k), (int,float)) and not isinstance(r.get(k),bool) for r in rows)]
            labels = [k for k in ('invoice_month', 'description', 'country', 'stock_code') if k in rows[0]]
            if not spec.get('x') and labels:
                spec['x'] = labels[0]
            if not spec.get('y') and len(numeric) == 1:
                spec['y'] = numeric[0]
            if any(spec.get(axis) not in rows[0] for axis in ('x','y')):
                problem = 'chart axes must be columns in the source result'
            elif spec['type'] != 'table' and spec['y'] not in numeric:
                problem = 'the chart y-axis must contain numerical measurements'
            elif spec.get('x') == spec.get('y'):
                problem = 'choose distinct chart axes'
            elif spec['type'] == 'pie' and (len(rows) < 2 or any(float(r[spec['y']]) < 0 for r in rows) or sum(float(r[spec['y']]) for r in rows) <= 0):
                problem = 'A share chart needs at least two non-negative categories with a positive total.'
    return add_to_log(log, 'make_chart', json.dumps(spec), not problem,
                      [spec] if not problem else [], error=problem)


## 5. Asking the model

We give Ollama a JSON schema, so the reply has to come back in the shape
we asked for. Much more reliable than asking nicely. If it still will not
parse, we try once more.


In [25]:
def model_usage(responses):
    """Keep actual Ollama token counts; missing usage is unknown, not zero."""
    if not responses:
        return {'model_calls': 0, 'input_tokens': 0, 'output_tokens': 0}
    return {key: sum(r[key] for r in responses) if all(r.get(key) is not None for r in responses) else None
            for key in ('model_calls', 'input_tokens', 'output_tokens')}


def ask_gemma(system, question, shape):
    """Ask the model for JSON of a particular shape.

    `shape` is a JSON schema. Ollama forces the reply to match it, which is a
    lot more reliable than asking nicely and hoping. If the reply still will
    not parse we try once more with a blunter instruction.
    """
    import ollama

    usage = []
    for attempt in range(2):
        prompt = question if attempt == 0 else question + """

YOUR LAST REPLY WAS NOT VALID JSON.
Reply with ONE complete JSON object matching the schema. Keep the text short.
Do not write anything outside the JSON object."""

        try:
            reply = ollama.Client(timeout=120).chat(model=MODEL, format=shape,
                                options={"temperature": 0, "num_predict": 2048},
                                messages=[{"role": "system", "content": system},
                                          {"role": "user", "content": prompt}])
        except ollama.ResponseError as error:
            if error.status_code != 500:
                raise  # Missing models/authentication need an actionable setup error.
            usage.append({'model_calls': 1, 'input_tokens': None, 'output_tokens': None})
            # The caller can retry planning/SQL within its normal budget. An
            # optional chart failure must not discard a successful data query.
            return {'broken_json': True, 'error': 'Model generation failed: ' + str(error),
                    '_usage': model_usage(usage)}
        usage.append({'model_calls': 1, 'input_tokens': reply.get('prompt_eval_count'),
                      'output_tokens': reply.get('eval_count')})
        try:
            parsed = json.loads(reply["message"]["content"])
            if isinstance(parsed, dict):
                parsed['_usage'] = model_usage(usage)
                return parsed
        except Exception:
            continue

    return {"broken_json": True, "_usage": model_usage(usage)}


### The shapes we ask for

One shape per step. Small shapes work better - a 4B model fills in three
fields reliably and fifteen fields badly.


In [ ]:
# The shapes we ask the model to fill in. Keeping them small is deliberate -
# a 4B model fills in three fields reliably and fifteen fields badly.

PLAN_SHAPE = {
    "type": "object",
    "properties": {
        "sufficient_data": {"type": "boolean"},
        "reason": {"type": "string"},
        "steps": {"type": "array", "items": {
            "type": "object",
            "properties": {
                "step": {"type": "integer"},
                "tool": {"type": "string", "enum": ["run_sql", "run_python", "make_chart"]},
                "objective": {"type": "string"},
            },
            "required": ["step", "tool", "objective"]}},
    },
    "required": ["sufficient_data", "steps"],
}

SQL_SHAPE = {"type": "object",
             "properties": {"sql": {"type": "string"}},
             "required": ["sql"]}

PYTHON_SHAPE = {
    "type": "object",
    "properties": {
        "operation": {"type": "string", "enum": OPERATIONS},
        "values": {"type": "array", "items": {"type": "number"}},
        "result_name": {"type": "string"},
        "unit": {"type": "string"},
    },
    "required": ["operation", "values", "result_name"],
}

CHART_SHAPE = {
    "type": "object",
    "properties": {
        "type": {"type": "string", "enum": CHART_TYPES},
        "source_tool_call": {"type": "integer"},
        "x": {"type": "string"},
        "y": {"type": "string"},
        "title": {"type": "string"},
    },
    "required": ["type", "source_tool_call", "x", "y", "title"],
}

ANSWER_SHAPE = {
    "type": "object",
    "properties": {
        "findings": {"type": "string"},
        "claims": {"type": "array", "items": {
            "type": "object",
            "properties": {
                "text": {"type": "string"},
                "value": {"type": "number"},
                "kind": {"type": "string", "enum": ["number", "boolean"]},
                "row": {"type": "integer"},
                "column": {"type": "string"},
                "unit": {"type": "string"},
                "from_call": {"type": "integer"},
                "calc": {"type": "string",
                         "enum": ["none", "pct_change", "share", "sum", "diff",
                                  "difference", "ratio", "mean"]},
                "inputs": {"type": "array", "items": {"type": "number"}},
            },
            "required": ["text", "value", "unit", "from_call", "calc", "inputs"]}},
        "kpis": {"type": "object"},
        "limitations": {"type": "string"},
        "insufficient_data": {"type": "boolean"},
    },
    "required": ["findings", "claims"],
}


## 6. The prompts

The examples matter more than the rules. Gemma will not remember
"exclude cancellations" from a sentence, but it will copy a query it has
just been shown.


In [29]:
SCHEMA = """
Database: a UK online gift wholesaler, Dec 2009 to 9 Dec 2011.

TABLE sales (one row per invoice line, not one row per order)
  invoice_no, stock_code, description, quantity, unit_price, customer_id,
  country, revenue (= quantity * unit_price), invoice_date, invoice_month,
  is_cancellation, is_product, is_outlier
  invoice_date is a timestamp. invoice_month is text 'YYYY-MM'.
  Country names are stored in full: UK is 'United Kingdom'.
  sales already contains customer_id, country and product details: no join is
  needed for customer or country rankings. NULL customer_id means a guest.

TABLE dim_month (one row per calendar month)
  invoice_month TEXT 'YYYY-MM', trading_days, net_revenue, gross_revenue,
  is_complete_month BOOLEAN
  net_revenue EXCLUDES cancellations. gross_revenue INCLUDES cancellations.
  Use net_revenue for ordinary monthly revenue and divide by trading_days
  for revenue per trading day. Do not apply sales-only fields to this table.

TABLE dim_product stock_code, description, units_sold, gross_revenue
TABLE dim_customer customer_id, country, first_order, last_order, orders, net_revenue
  These two dimension summaries cover ALL dates; they cannot answer a year's
  totals. dim_customer.net_revenue includes cancellations despite its name.
  Use sales for revenue, order and customer analyses.

There is NO cost, profit, margin, discount, competitor or customer age data.
"""

RULES = """
BUSINESS RULES
1. Overall, country and customer revenue: SUM(revenue) FROM sales WHERE
   NOT is_cancellation. Keep every remaining line: do NOT add is_product,
   is_outlier or quantity filters. Keep guests for overall/country revenue.
2. ONLY product-specific revenue questions add is_product AND NOT is_outlier.
   Group by stock_code; use mode(description) for the name. Units sold needs
   quantity > 0; do not apply that condition to revenue or completed orders.
3. Completed orders = COUNT(DISTINCT invoice_no), NOT COUNT(*) or line counts,
   with NOT is_cancellation. Customer rankings also need customer_id IS NOT NULL
   and GROUP BY customer_id. Customer IDs are labels, not measurements.
4. For monthly totals, rankings, comparisons and trading days use dim_month.
   Select invoice_month as a label and trading_days as context. Complete-month
   revenue rankings require WHERE is_complete_month. Monthly growth is
   100.0 * (new - old) / NULLIF(old,0); use unrounded inputs until the final ROUND.
   Per-day growth first divides EACH month's net_revenue by its trading_days.
5. December 2011 is incomplete; never compare it as a full month.
6. A share needs a part and the WHOLE population: do not restrict the denominator
   to the chosen country. Use country = 'United Kingdom' for UK.
7. For an extreme value, return every tied winner unless a fixed list size is
   requested. Use a maximum/minimum subquery or DENSE_RANK; LIMIT 1 hides ties.
8. A year applies to every sales query for that question. Use year(invoice_date)
   or timestamp bounds; dim_month uses 'YYYY-MM' text bounds/IN. Each SELECT
   must use the schema of its own table. Prefer one simple grouped query over
   unnecessary joins. Never join sales to monthly totals and then SUM those totals.
9. Never say what CAUSED something, only what contributed to it. If the data
   cannot answer the question, explain what is missing instead of guessing.
"""

EXAMPLES = """
SQL PATTERNS (adapt the country, period, metric and limit to the QUESTION)

Country revenue ranking, no product filters or customer joins:
  SELECT country, ROUND(SUM(revenue),2) AS revenue FROM sales
  WHERE NOT is_cancellation AND year(invoice_date) = 2010
  GROUP BY country ORDER BY revenue DESC LIMIT 3

Identified customer order ranking, one row per customer:
  SELECT customer_id, COUNT(DISTINCT invoice_no) AS orders FROM sales
  WHERE NOT is_cancellation AND customer_id IS NOT NULL
    AND year(invoice_date) = 2010
  GROUP BY customer_id ORDER BY orders DESC LIMIT 3

A country's share with an unrestricted denominator:
  SELECT SUM(CASE WHEN country = 'Sweden' THEN revenue ELSE 0 END) AS country_revenue,
         SUM(revenue) AS total_revenue,
         100.0 * SUM(CASE WHEN country = 'Sweden' THEN revenue ELSE 0 END)
           / NULLIF(SUM(revenue),0) AS share_pct
  FROM sales WHERE NOT is_cancellation AND year(invoice_date) = 2010

Monthly comparison evidence (then calculate the requested change):
  SELECT invoice_month, net_revenue AS revenue, trading_days,
         net_revenue / NULLIF(trading_days,0) AS revenue_per_day, is_complete_month
  FROM dim_month WHERE invoice_month IN ('2010-05','2010-06')
  ORDER BY invoice_month

All tied extrema, with the same scope in the inner and outer query:
  SELECT invoice_month, trading_days FROM dim_month
  WHERE trading_days = (SELECT MIN(trading_days) FROM dim_month)
  ORDER BY invoice_month
"""

SYSTEM = "You are a careful business data analyst.\n" + SCHEMA + RULES

PLAN_PROMPT = SYSTEM + """
Plan how to answer the question. Do NOT answer it - you have not seen any data.

List the steps in order. Each step uses one tool:
  run_sql      fetch numbers from the database
  run_python   work something out from numbers a query already returned
  make_chart   show the result

Use the shortest useful plan. A total usually needs only one SQL query.
Do not add arithmetic when SQL already returned the answer. Only add a chart
when the question asks for one or when comparing several categories or periods.
Set sufficient_data to false if the data cannot answer the question at all.
"""

ANSWER_PROMPT = SYSTEM + """
You asked for some queries and here are the results. Write the answer using
ONLY these numbers.

Every number in findings must also appear as a claim, with:
  from_call  which call it came from
  calc       none, unless you worked it out - then pct_change/share/sum/diff/ratio
  inputs     the numbers you worked it out from

Write plain business prose in findings. Do not include call IDs, calculation
labels, or input lists in the prose; those belong in claim fields.
State that revenue excludes cancellations. Use digits for numerical claims.
For yes/no facts, leave numerical claims empty; the tool evidence supplies
typed boolean facts. Never encode a boolean as 0 or 1.
For numeric facts include row and column when copied from SQL. Keep every
product and its value in a separate claim. Preserve product names exactly.
Do not say what caused anything. Say what contributed.
"""


def sql_prompt(question, objective, feedback=''):
    """The caller supplies SYSTEM once; keep user-side SQL guidance focused."""
    return (EXAMPLES
            + '\nQUESTION\n' + question
            + '\nRELEVANT RULES\n' + query_guidance(question)
            + ('\nPREVIOUS ATTEMPT FEEDBACK — correct these issues\n' + str(feedback) if feedback else '')
            + '\nWRITE ONE SELECT QUERY FOR THIS STEP ONLY\n' + str(objective)
            + '\nUse exact schema columns, unique output aliases and the requested population. Return SQL in the required JSON field.')


def repair_prompt(question, objective, sql, error, feedback=''):
    """Give a small model one concrete edit, without repeating schema/examples."""
    message = str(error)
    normalized = re.sub(r'\b\w+\.', '', str(sql).lower())
    normalized = re.sub(r'\s+', ' ', normalized)
    instructions = []
    if 'Customer rankings require' in message:
        if not re.search(r'customer_id is not null|not customer_id is null', normalized):
            instructions.append('Add AND customer_id IS NOT NULL to the existing WHERE clause, before GROUP BY.')
        try:
            tree = sqlglot.parse_one(str(sql), read='duckdb')
            grouped = any(col.name.lower() == 'customer_id' for group in tree.find_all(exp.Group)
                          for col in group.find_all(exp.Column))
        except Exception:
            grouped = False
        if not grouped:
            instructions.append('Select customer_id and group by customer_id; do not group a customer ranking by country.')
        instructions.append('Keep the existing requested year, metric, ordering and limit.')
    elif 'Completed orders require' in message:
        instructions.append('Use COUNT(DISTINCT invoice_no) for the orders result. Keep NOT is_cancellation. Remove quantity, is_product and is_outlier filters.')
    elif 'must not use product-only' in message:
        instructions.append('Delete is_product, is_outlier and quantity conditions from WHERE. Keep NOT is_cancellation and the requested date/country/customer conditions.')
    elif "database has no country named 'UK'" in message:
        instructions.append("Replace the country literal 'UK' with 'United Kingdom'; preserve the rest of the query.")
    elif 'Complete' in message or 'complete_month' in message:
        instructions.append('Add is_complete_month to the dim_month WHERE clause. Keep the requested months, revenue metric and ordering.')
    else:
        instructions.append('Correct the error below. Preserve the requested dates, labels and metrics; remove rejected filters. Use only columns belonging to the named table.')
    return ('QUESTION\n' + str(question)
            + '\nSTEP\n' + str(objective)
            + '\nFAILED SQL\n' + str(sql)
            + '\nERROR\n' + message
            + ('\nVERIFIER FEEDBACK\n' + str(feedback) if feedback else '')
            + '\nREQUIRED EDIT\n' + '\n'.join(instructions)
            + '\nReturn one corrected SELECT in the required JSON sql field. Apply the edit; do not repeat the failed query unchanged.')


def python_prompt(question, objective, log):
    return (SYSTEM
            + "\nQUESTION\n" + question
            + "\n\nSTEP\n" + objective
            + "\n\nNUMBERS THE QUERIES RETURNED\n" + show_results(log)
            + "\nPick the operation and give us the numbers. We do the arithmetic.")


def chart_prompt(question, objective, log):
    return (SYSTEM
            + "\nQUESTION\n" + question
            + "\n\nSTEP\n" + objective
            + "\n\nRESULTS SO FAR\n" + show_results(log)
            + "\nChoose a chart. source_tool_call is the call number to plot.")


def answer_prompt(question, log, feedback=""):
    text = ("QUESTION\n" + question
            + "\n\nTOOL RESULTS\n" + show_results(log))
    if feedback:
        text += "\n\nYOUR LAST ANSWER FAILED CHECKING\n" + feedback
    return text


## 7. Two guards and a safety net

`touches_incomplete_month` stops us working out a month-over-month change
when one of the months is December 2011, which only has 8 trading days.

`claims_from_log` builds the claims ourselves if the model forgets to list
them. Without it, an answer with no claims would sail through Tier 3 with
nothing checked.


In [31]:
def touches_incomplete_month(log):
    """Did any query return a month flagged as incomplete?"""
    for call in log:
        if not call["ok"]:
            continue
        for row in call["rows"]:
            if row.get("is_complete_month") is False:
                return True
    return False


def mentions_december_2011(question):
    text = question.lower()
    return ("december 2011" in text or "dec 2011" in text
            or "2011-12" in text)


def claims_from_log(log):
    """Preserve typed facts and their row labels when the writer omits claims."""
    claims = []
    for call in log:
        if not call.get('ok') or call.get('tool') == 'make_chart':
            continue
        for index, row in enumerate(call['rows']):
            if call['tool'] == 'run_python':
                claims.append({'text': str(row.get('result_name', 'Calculation')) + ': ' + str(row['value']),
                               'value': row['value'], 'unit': row.get('unit'), 'from_call': call['n'],
                               'calc': row['operation'], 'inputs': row['inputs']})
                continue
            label = str(row.get('description') or row.get('country') or row.get('invoice_month') or '')
            for key, value in row.items():
                if key in ('customer_id', 'stock_code') or not isinstance(value, (int,float,bool)):
                    continue
                if isinstance(value, bool):
                    text = ('The month is complete.' if value else 'The month is incomplete.') if key == 'is_complete_month' else key + (' is true.' if value else ' is false.')
                    claims.append(evidence_claim(call, index, key, text))
                    continue
                unit = {'revenue':'GBP','net_revenue':'GBP','aov':'GBP','trading_days':'days','units':'units'}.get(key,'')
                text = (label + ': ' if label else '') + key.replace('_',' ') + ' = ' + str(value)
                claims.append(evidence_claim(call, index, key, text, unit))
    return claims


## 8. Tier 2


In [33]:
def data_gap_reason(question):
    """Cheap scope checks for known absent data; keep the weak planner from guessing."""
    text = question.lower()
    missing = r"\b(profit|profits|margin|margins|cost|costs|cogs|discounts?|promotions?|campaigns?|marketing|demographics?|competitors?|customer age|web traffic|website traffic)\b"
    if re.search(missing, text):
        return "This dataset contains sales transactions, but not the cost, promotion or other external data needed to answer this question."
    if re.search(r"\b(forecast|predict|prediction|next quarter|next year|next month)\b", text):
        return "Forecasting is outside this project's scope. These historical transactions cannot establish future revenue."
    comparison = re.search(r"\b(compare|compared|comparison|versus|vs|beat|change|growth|decline|increase|decrease|difference|drop)\b", text)
    normalized = "per trading day" in text or "per day" in text
    if mentions_december_2011(question) and comparison and not normalized:
        return "December 2011 is an incomplete reporting month. A full-month comparison would be misleading; ask for individual totals or revenue per trading day."
    return None


def question_contract(question):
    """Compile a small, explicit vocabulary of questions into KPI-safe queries.

    Match the whole question so an extra country/product filter is never lost.
    These are reusable query patterns, not benchmark answers: dates and limits
    come from the question, and every value still comes from the live database.
    Unrecognised wording continues through the model-driven tool loop.
    """
    import calendar
    text = re.sub(r"\s+", " ", question.lower()).strip().rstrip("?.")
    year = r"(?P<year>(?:19|20)\d{2})"
    count = r"(?P<count>\d+|one|two|three|four|five|six|seven|eight|nine|ten)"
    names = {name.lower(): i for i, name in enumerate(calendar.month_name) if name}
    month = "(?:" + "|".join(names) + ")"
    result = None
    patterns = [
        (rf"(?:what was (?:our |the )?(?:total )?revenue|(?:total )?revenue) in {year}", "revenue"),
        (rf"how many customers (?:bought from us|purchased) in {year}", "customers"),
        (rf"how many orders did we (?:take|receive) in {year}", "orders"),
        (rf"how many units did we sell in {year}", "units"),
        (rf"what was (?:our |the )?average order value in {year}", "aov"),
        (rf"what was (?:our |the )?cancellation rate in {year}", "cancellation_rate"),
        (rf"(?:list the {count} products that generated the most revenue in {year},? with the revenue for each)", "products"),
        (rf"top {count} products (?:by revenue )?in {year}", "products"),
        (rf"which {count} products generated the most revenue in {year}", "products"),
        (rf"which {count} products sold the most units in {year}", "product_units"),
    ]
    for pattern, kind in patterns:
        match = re.fullmatch(pattern, text)
        if not match:
            continue
        y = match['year']
        where = f"NOT is_cancellation AND year(invoice_date) = {y}"
        if kind == 'cancellation_rate':
            where = f"year(invoice_date) = {y}"
        expressions = {'cancellation_rate': 'ROUND(100.0 * COUNT(DISTINCT CASE WHEN is_cancellation THEN invoice_no END) / NULLIF(COUNT(DISTINCT CASE WHEN NOT is_cancellation THEN invoice_no END),0),2)',
                       'revenue': 'ROUND(SUM(revenue),2)',
                       'customers': 'COUNT(DISTINCT customer_id)',
                       'orders': 'COUNT(DISTINCT invoice_no)',
                       'units': 'SUM(quantity)',
                       'aov': 'ROUND(SUM(revenue)/NULLIF(COUNT(DISTINCT invoice_no),0),2)'}
        if kind in ('products', 'product_units'):
            words = 'zero one two three four five six seven eight nine ten'.split()
            raw = match['count']
            n = int(raw) if raw.isdigit() else words.index(raw)
            if not 1 <= n <= 50:
                return None
            metric = 'revenue' if kind == 'products' else 'units'
            if metric == 'units':
                where += ' AND quantity > 0'
            sql = (f"SELECT stock_code, mode(description) AS description, {expressions[metric]} AS {metric} "
                   f"FROM sales WHERE {where} AND is_product AND NOT is_outlier "
                   f"GROUP BY stock_code ORDER BY {metric} DESC, stock_code LIMIT {n}")
            result = {'kind': kind, 'metric': metric, 'count': n, 'year': y, 'sql': sql}
        else:
            if kind == 'customers':
                where += ' AND customer_id IS NOT NULL'
            if kind == 'units':
                where += ' AND quantity > 0'
            result = {'kind': kind, 'metric': kind, 'year': y,
                      'sql': f"SELECT {expressions[kind]} AS {kind} FROM sales WHERE {where}"}
        break
    if result:
        return result
    foreign = re.fullmatch(rf"which country outside (?:the )?(?:uk|united kingdom) generated the most revenue in {year},? and how much was it", text)
    if foreign:
        return {'kind': 'country_rank', 'metric': 'revenue', 'year': foreign['year'],
                'sql': "SELECT country, ROUND(SUM(revenue),2) AS revenue FROM sales WHERE NOT is_cancellation "
                       f"AND year(invoice_date) = {foreign['year']} AND country <> 'United Kingdom' "
                       "GROUP BY country ORDER BY revenue DESC, country LIMIT 1"}
    match = re.fullmatch(rf"is (?P<month>{month}) {year} (?:a )?complete month(?: in the dataset)?", text)
    if match:
        period = f"{match['year']}-{names[match['month']]:02d}"
        return {'kind': 'completeness', 'period': period,
                'sql': f"SELECT invoice_month, trading_days, is_complete_month FROM dim_month WHERE invoice_month = '{period}'"}
    match = re.fullmatch(rf"did (?P<new>{month}) (?P<newyear>(?:19|20)\d{{2}}) beat (?P<old>{month}) (?P<oldyear>(?:19|20)\d{{2}}) on revenue,? and by what percentage", text)
    if match:
        new = f"{match['newyear']}-{names[match['new']]:02d}"
        old = f"{match['oldyear']}-{names[match['old']]:02d}"
        if new == old:
            return None
        return {'kind': 'comparison', 'new': new, 'old': old,
                'sql': "SELECT invoice_month, ROUND(net_revenue,2) AS revenue, trading_days, is_complete_month "
                       f"FROM dim_month WHERE invoice_month IN ('{old}', '{new}') ORDER BY invoice_month"}
    return None


def evidence_claim(call, row_index, column, text, unit=''):
    """Bind a fact to a specific cell, keeping labels and measurements together."""
    value = call['rows'][row_index][column]
    return {'text': text, 'value': value, 'unit': unit, 'from_call': call['n'],
            'calc': 'none', 'inputs': [], 'row': row_index, 'column': column,
            'kind': 'boolean' if isinstance(value, bool) else 'number'}


def patterned_answer(question, contract, log):
    """Execute common analyses without making a small model copy or invent facts.

    Shared by Tier 2 and Tier 3. Only Tier 3 applies independent verification.
    Keep the query, calculation and chart calls in the same evidence log as the
    model-driven path, so this optimisation stays visible in the evaluation.
    """
    start = time.time()
    call = run_sql(contract['sql'], log, question)
    fields, filters = sql_metadata(contract['sql'])
    answer = {'question': question, 'tier': 2, 'findings': '', 'claims': [], 'kpis': {},
              'fields_used': fields, 'filters_used': filters, 'chart': None,
              'insufficient_data': False, 'limitations': '', 'log': log, 'retries': 0,
              'plan': ['run_sql: use the matching KPI query pattern'], 'seconds': 0,
              'route': 'query_pattern', 'scope_check': 'matching KPI query pattern', 'usage': model_usage([])}
    rows = call['rows']
    if not call['ok'] or not rows or any(v is None for r in rows for v in r.values()):
        answer.update(findings='No usable data was returned for this request.',
                      limitations=call.get('error') or 'The requested period has no data.')
        return answer
    claims = answer['claims']
    kind = contract['kind']
    spec = None
    if kind in ('products', 'product_units', 'country_rank'):
        metric = contract['metric']
        for i, row in enumerate(rows):
            amount = f"£{row[metric]:,.2f}" if metric == 'revenue' else f"{row[metric]:,} units"
            claims.append(evidence_claim(call, i, metric, f"{row.get('description', row.get('country'))}: {amount}.",
                                         'GBP' if metric == 'revenue' else 'units'))
        spec = {'type': 'bar', 'source_tool_call': call['n'], 'x': 'country' if kind == 'country_rank' else 'description',
                'y': metric, 'title': 'Leading market' if kind == 'country_rank' else 'Leading products'}
        answer['limitations'] = ('Cancellations and the United Kingdom are excluded.' if kind == 'country_rank' else 'Cancellations, non-product lines and flagged outliers are excluded.')
    elif kind == 'completeness':
        row = rows[0]
        label = 'complete' if row['is_complete_month'] else 'incomplete'
        claims.append(evidence_claim(call, 0, 'is_complete_month', f"The month is {label}."))
        claims.append(evidence_claim(call, 0, 'trading_days', f"It contains {row['trading_days']} trading days.", 'days'))
        answer['limitations'] = 'An incomplete month cannot be compared as a full month.' if not row['is_complete_month'] else ''
    elif kind == 'comparison':
        by_month = {r['invoice_month']: i for i, r in enumerate(rows)}
        if set(by_month) != {contract['new'], contract['old']}:
            answer.update(findings='Both requested months must have data before they can be compared.',
                          limitations='One or more reporting months are missing.')
            return answer
        if not all(r['is_complete_month'] for r in rows):
            answer.update(findings='A requested month is incomplete, so a full-month comparison would be misleading.',
                          insufficient_data=True, limitations='Ask for revenue per trading day instead.')
            return answer
        new, old = (rows[by_month[contract[k]]]['revenue'] for k in ('new', 'old'))
        calculation = run_python({'operation': 'pct_change', 'values': [new, old], 'unit': '%',
                                  'result_name': 'Revenue change'}, log)
        for i, row in enumerate(rows):
            claims.append(evidence_claim(call, i, 'revenue', f"In {row['invoice_month']}, revenue was £{row['revenue']:,.2f}.", 'GBP'))
            claims.append(evidence_claim(call, i, 'trading_days', f"In {row['invoice_month']}, there were {row['trading_days']} trading days.", 'days'))
        if calculation['ok']:
            value = calculation['rows'][0]['value']
            text = ('Yes, revenue increased' if value > 0 else 'No, revenue decreased' if value < 0 else 'No, revenue was unchanged')
            claims.append({'text': f"{text} by {abs(value):.2f}%." if value >= 0 else f"No, revenue changed by {value:.2f}%.",
                           'value': value, 'unit': '%', 'from_call': calculation['n'], 'calc': 'pct_change',
                           'inputs': [new, old], 'input_cells': [{'from_call': call['n'], 'row': by_month[contract[k]], 'column': 'revenue'} for k in ('new', 'old')]})
        else:
            answer['limitations'] = 'Percentage change is undefined when the earlier revenue is zero. '
        answer['limitations'] += 'Revenue excludes cancellations. Trading-day counts are shown for context.'
        spec = {'type': 'line', 'source_tool_call': call['n'], 'x': 'invoice_month', 'y': 'revenue', 'title': 'Monthly revenue'}
    else:
        metric = contract['metric']
        value = rows[0][metric]
        if metric == 'cancellation_rate':
            text = f"Cancellation invoices were {value:.2f}% of completed orders."
            unit = '%'
        elif metric in ('revenue', 'aov'):
            text = f"{'Revenue' if metric == 'revenue' else 'Average order value'} was £{value:,.2f}."
            unit = 'GBP'
        else:
            text = f"There were {value:,} {'identified customers' if metric == 'customers' else metric}."
            unit = metric
        claims.append(evidence_claim(call, 0, metric, text, unit))
        answer['limitations'] = 'Cancellations are excluded.'
        if metric == 'cancellation_rate':
            answer['limitations'] = 'Cancellation invoices cannot be linked to their original orders. This is a ratio of cancellations to completed orders in the same period, not the share of orders later cancelled.'
        if metric == 'customers':
            answer['limitations'] += ' Transactions without a customer ID are excluded from this customer count.'
    if spec:
        chart_call = make_chart(spec, log)
        if chart_call['ok']:
            answer['chart'] = chart_call['rows'][0]
    answer['findings'] = ' '.join(c['text'] for c in claims)
    if kind not in ('completeness', 'cancellation_rate'):
        answer['findings'] += ' Cancellations are excluded.'
    answer['seconds'] = round(time.time() - start, 2)
    return answer


def label_spans(text, answer):
    """Only exact categorical labels in successful SQL evidence exempt digits.

    Never exempt numeric strings, arbitrary text columns or all small numbers.
    That would allow an unsupported count to slip through the verifier.
    """
    spans = []
    for call in answer.get('log', []):
        if call.get('ok') and call.get('tool') == 'run_sql':
            # A filtered identifier is also a label when explicitly introduced
            # as 'customer' or 'stock code'; it is not an asserted quantity.
            try:
                tree = sqlglot.parse_one(call.get('code', ''), read='duckdb')
                for where in tree.find_all(exp.Where):
                    for predicate in where.find_all(exp.EQ):
                        column, literal = predicate.this, predicate.expression
                        if isinstance(column, exp.Column) and isinstance(literal, exp.Literal):
                            prefix = {'customer_id': r'customer(?:[ _]id)?', 'stock_code': r'(?:stock|product) code'}.get(column.name.lower())
                            if prefix:
                                pattern = r'\b' + prefix + r'\s+' + re.escape(str(literal.this)) + r'(?!\w)'
                                spans += [(m.start(),m.end()) for m in re.finditer(pattern,text,re.I)]
            except Exception:
                pass  # Malformed SQL never creates a label exemption.
            for row in call.get('rows', []):
                for key in ('description', 'stock_code', 'customer_id', 'country', 'invoice_month'):
                    label = str(row.get(key) or '')
                    # DuckDB stores nullable customer IDs as doubles. Rendered
                    # integer IDs retain their identity, not a new quantity.
                    if key == 'customer_id' and re.fullmatch(r'\d+\.0+', label):
                        label = label.split('.')[0]
                    if not label or (key not in ('stock_code', 'customer_id') and not re.search(r'[A-Za-z-]', label)):
                        continue
                    # A numerical code needs an explicit identifier prefix.
                    pattern = re.escape(label)
                    if label.isdigit():
                        if key == 'customer_id':
                            pattern += r'(?:\.0+)?'
                        pattern = (r'customer(?:[ _]id)?\s+' if key == 'customer_id' else r'(?:stock code\s+|product code\s+)') + pattern
                    spans += [(m.start(), m.end()) for m in re.finditer(r'(?<!\w)' + pattern + r'(?!\w)', text, re.I)]
    return spans


def is_evidence_label(text, start, end, answer):
    return any(left <= start and end <= right for left, right in label_spans(text, answer))


def claim_cell(claim, log):
    """Resolve an explicit cell reference without Python's False == 0 shortcut."""
    source = next((c for c in log if c.get('n') == claim.get('from_call') and c.get('ok') and c.get('tool') == 'run_sql'), None)
    index, column = claim.get('row'), claim.get('column')
    if not source or type(index) is not int or not 0 <= index < len(source.get('rows', [])):
        return False, None
    row = source['rows'][index]
    return (True, row[column]) if column in row else (False, None)


def tier2(question, log=None, feedback=""):
    """Plan, run the steps, then write the answer from the real rows.

    The model never sees the database and never writes Python. It says what it
    wants; we run it and record what happened.
    """
    log = log if log is not None else []
    start = time.time()
    first_call = len(log)
    usage = []
    def model_call(system, prompt, shape):
        response = ask_gemma(system, prompt, shape)
        usage.append(response.get('_usage', {}))
        return response

    def stop(findings, limitations=None, insufficient=False, plan_steps=None):
        return {"question": question, "tier": 2, "findings": findings,
                "claims": [], "kpis": {}, "fields_used": [], "filters_used": [],
                "chart": None, "insufficient_data": insufficient,
                "limitations": limitations, "log": log, "retries": 0,
                "plan": plan_steps or [], "route": "model_tools",
                "failure": None if insufficient else (limitations or findings), "usage": model_usage(usage), "seconds": round(time.time() - start, 1)}

    gap = data_gap_reason(question)
    if gap:
        answer = stop(gap, gap, True)
        answer["route"] = "rules_refusal"
        return answer

    contract = question_contract(question)
    if contract:
        return patterned_answer(question, contract, log)

    # ---- 1. plan
    ask = question if not feedback else question + "\n\nLast attempt failed:\n" + feedback
    plan = model_call(PLAN_PROMPT, ask, PLAN_SHAPE)

    if plan.get("broken_json"):
        return stop("The planner did not return valid JSON.",
                    plan.get("error") or "the model did not return valid JSON")

    steps = [s for s in (plan.get("steps") or []) if isinstance(s, dict)]
    steps = sorted(steps, key=lambda s: s.get("step") if isinstance(s.get("step"), (int, float)) else 0)
    plan_text = ["%s: %s" % (s.get("tool"), s.get("objective")) for s in steps]

    # A small model often sets sufficient_data to false while still listing
    # steps it wants to run. gemma3:4b did exactly that on "what was our total
    # revenue in 2011", and Tier 2 refused a question it could easily answer.
    #
    # So we treat the STEPS as the real signal. If it asked for queries, it
    # thinks it can answer. We only refuse when it says no AND gives us
    # nothing to run.
    says_no = not plan.get("sufficient_data", True)

    if says_no and not steps:
        reason = plan.get("reason") or "The data cannot answer this question."
        return stop(reason, reason, True, plan_text)

    if not steps:
        return stop("The planner produced no steps to run.",
                    "the planner marked the question answerable but gave no steps",
                    False, plan_text)

    # ---- 2. run the steps
    chart = None

    for step in steps[:6]:
        tool = step.get("tool")
        objective = step.get("objective", "")

        if tool == "run_sql":
            request = model_call(SYSTEM, sql_prompt(question, objective, feedback), SQL_SHAPE)
            call = run_sql(request.get("sql", ""), log, question)

            # one repair attempt, with the error message
            if not call["ok"]:
                fix = model_call(SYSTEM,
                                repair_prompt(question, objective,
                                              call["code"], call["error"], feedback),
                                SQL_SHAPE)
                call = run_sql(fix.get("sql", ""), log, question)

            if not call["ok"]:
                return stop("The query could not be made to run.",
                            call["error"], False, plan_text)

        elif tool == "run_python":
            # Do not work out a month-over-month change when one of the months
            # is incomplete. December 2011 has 8 trading days.
            if touches_incomplete_month(log) or mentions_december_2011(question):
                continue
            request = model_call(SYSTEM, python_prompt(question, objective, log),
                                PYTHON_SHAPE)
            run_python(request, log)

        elif tool == "make_chart":
            spec = model_call(SYSTEM, chart_prompt(question, objective, log),
                             CHART_SHAPE)
            call = make_chart(spec, log)
            if call["ok"] and spec.get("type") != "none":
                chart = call["rows"][0]

    # ---- 3. refuse if nothing worked
    useful = [c for c in log[first_call:]
              if c["ok"] and c["tool"] in ("run_sql", "run_python")]
    if not useful:
        # Nothing worked. Refusing is the honest outcome - we have no numbers,
        # so any answer would be invented.
        return stop("No query produced any evidence, so there is no answer to give.",
                    "no successful query or calculation", True, plan_text)

    # ---- 4. write the answer from the real rows
    reply = model_call(ANSWER_PROMPT, answer_prompt(question, log, feedback),
                      ANSWER_SHAPE)

    if reply.get("broken_json"):
        return stop("The model did not return a usable answer.",
                    reply.get("error") or "the model did not return valid JSON", False, plan_text)

    claims = reply.get("claims") or []
    if not isinstance(claims, list):
        claims = []
    claims = [c for c in claims if isinstance(c, dict)]
    if not claims:
        claims = claims_from_log(log[first_call:])

    # Boolean facts have a separate typed route; never coerce False to zero.
    for fact in claims_from_log(log[first_call:]):
        if fact.get('kind') == 'boolean':
            if not any(c.get('kind') == 'boolean' and c.get('from_call') == fact['from_call'] and c.get('column') == fact['column'] and c.get('row') == fact['row'] for c in claims):
                claims.append(fact)

    # ---- 5. fields and filters come from the SQL, not from the model
    fields, filters = [], []
    for call in log[first_call:]:
        if call["tool"] != "run_sql" or not call["ok"]:
            continue
        f, w = sql_metadata(call["code"])
        fields += [x for x in f if x not in fields]
        filters += [x for x in w if x not in filters]

    limitations = reply.get("limitations")
    if (touches_incomplete_month(log) or mentions_december_2011(question)) \
            and not limitations:
        limitations = "An incomplete month is not comparable with a complete month."

    return {"question": question, "tier": 2,
            "findings": reply.get("findings", ""),
            "claims": claims,
            "kpis": reply.get("kpis") or {},
            "fields_used": fields or [],
            "filters_used": filters or [],
            "chart": chart,
            "route": "model_tools", "scope_check": "numeric evidence only",
            "insufficient_data": bool(reply.get("insufficient_data")),
            "limitations": limitations,
            "log": log, "retries": 0, "plan": plan_text, "usage": model_usage(usage),
            "seconds": round(time.time() - start, 1)}



## 9. Run it

Make sure Ollama is running: `ollama pull gemma3:4b`


In [18]:
answer = tier2('What was our total revenue in 2011?')

print('PLAN:')
for step in answer['plan']:
    print(' -', step)

print()
print('FINDINGS:', answer['findings'])
print('FIELDS  :', answer['fields_used'])
print('FILTERS :', answer['filters_used'])


PLAN:
 - run_sql: Calculate the total revenue for 2011, excluding cancellations and outliers, using the sales table.  Filter by invoice_month within the range of 2011 and using the rule that revenue excludes cancellations.
 - run_python: Aggregate the results from the SQL query to calculate the sum of the revenue column.
 - make_chart: Present the total revenue in 2011 as a numerical value.

FINDINGS: The data cannot answer this question.
FIELDS  : []
FILTERS : []


### What did it actually run?


In [19]:
for call in answer['log']:
    print('[%d] %s   ok=%s' % (call['n'], call['tool'], call['ok']))
    print(call['code'])
    print('->', call['rows'] if call['ok'] else call['error'])
    print()


### The claims

Every number listed separately, with where it came from. Notebook 05
needs this - you cannot reliably pull numbers out of a paragraph.


In [20]:
for c in answer['claims']:
    print(c.get('value'), c.get('unit'), '|', c.get('text'))
    print('    from call', c.get('from_call'), '| calc:', c.get('calc'))


## 10. Does it refuse impossible questions?

There is no cost data, so profit margin cannot be worked out.


In [21]:
bad_q = tier2('What was our profit margin in 2011?')

print('refused?    ', bad_q['insufficient_data'])
print('queries run:', len(bad_q['log']))
print('says:       ', bad_q['findings'])


refused?     True
queries run: 0
says:        The data cannot answer this question.


## 11. The harder ones


In [22]:
for q in ['Did November 2011 beat October 2011 on revenue, and by what percentage?',
          'List the five products that generated the most revenue in 2011, with the revenue for each.']:
    print('=' * 70)
    print(q)
    print('=' * 70)
    a = tier2(q)
    print(a['findings'])
    print()


Did November 2011 beat October 2011 on revenue, and by what percentage?
The data cannot answer this question.

List the five products that generated the most revenue in 2011, with the revenue for each.
The data cannot answer this question.



## What we found

Write this down, it goes in the report:

- did it remember `NOT is_cancellation`?
- did any query fail, and did the repair attempt fix it?
- did it fill in `calc` and `inputs` for calculated numbers?

**When Gemma writes a bad query, fix the prompt, not the code.** Add another
worked example to `EXAMPLES`.

The numbers are real now. But nothing checks that the *sentence* matches
the rows - that is notebook 05.


## Regression examples after the reliability fixes
These common analyses use checked query patterns in both grounded tiers. Tier 3 additionally verifies the returned facts. Free-form questions still use Gemma.


In [ ]:
# These checks query the real database; no model or benchmark answers are loaded.
example_questions = [
    "What was our total revenue in 2011?",
    "Which five products generated the most revenue in 2011?",
    "Did November 2011 beat October 2011 on revenue, and by what percentage?",
    "Is December 2011 a complete month in the dataset?",
    "What was our cancellation rate in 2011?",
]
for question in example_questions:
    result = tier2(question)
    print(question)
    print(result['findings'])
    print("Claims:", len(result['claims']), "| Chart:", (result['chart'] or {}).get('type', 'none'))
    print()


What was our total revenue in 2011?
Revenue was £9,809,614.01. Cancellations are excluded.
Claims: 1 | Chart: none

Which five products generated the most revenue in 2011?
REGENCY CAKESTAND 3 TIER: £146,461.78. PARTY BUNTING: £98,237.49. WHITE HANGING HEART T-LIGHT HOLDER: £94,027.39. JUMBO BAG RED RETROSPOT: £90,140.66. RABBIT NIGHT LIGHT: £66,870.03. Cancellations are excluded.
Claims: 5 | Chart: bar

Did November 2011 beat October 2011 on revenue, and by what percentage?
In 2011-10, revenue was £1,151,263.73. In 2011-10, there were 26 trading days. In 2011-11, revenue was £1,503,866.78. In 2011-11, there were 26 trading days. Yes, revenue increased by 30.63%. Cancellations are excluded.
Claims: 5 | Chart: line

Is December 2011 a complete month in the dataset?
The month is incomplete. It contains 8 trading days.
Claims: 2 | Chart: none

What was our cancellation rate in 2011?
Cancellation invoices were 17.24% of completed orders.
Claims: 1 | Chart: none

